# Práctica guiada · Tool Use

Este notebook está pensado para hacerse **después de la píldora**.

No vas a escribir bloques grandes de código. En cada paso completarás **una palabra, un valor, un nombre de función o un argumento**. Después de cada ejercicio hay una comprobación automática para que, si algo no coincide, el error aparezca **en ese mismo paso** y con un mensaje comprensible.

## Objetivo
Al terminar habrás construido un flujo completo:

**tool call → validación → selección de herramienta → ejecución → resultado**

> Idea clave: **el modelo solicita una herramienta; el software valida y la ejecuta.**

## 1. Creamos una herramienta real

Una *tool* puede ser una función Python normal. Nuestra primera herramienta consulta una agenda.

Fíjate en algo importante: la función recibe un parámetro llamado **`fecha`**. Más adelante, la tool call enviará también un argumento llamado **`fecha`**. Esos nombres deben coincidir.

In [ ]:
eventos = {
    "2026-10-26": ["09:30 Mentoría", "18:00 Bootcamp"],
    "2026-10-27": ["11:00 Reunión de proyecto"]
}

def consultar_agenda(fecha):
    return eventos.get(fecha, [])

print("Herramienta creada correctamente.")

### Ejercicio 1 · Usar la función

Completa solo el valor de `fecha_consulta` para consultar el **26 de octubre de 2026**.

**Pista:** la fecha debe escribirse exactamente como aparece en el diccionario `eventos`: `AAAA-MM-DD`.

In [ ]:
fecha_consulta = ""  # COMPLETA AQUÍ

assert fecha_consulta == "2026-10-26", (
    "Revisa fecha_consulta. Debe ser la fecha 2026-10-26."
)

resultado_directo = consultar_agenda(fecha_consulta)
print(resultado_directo)

### ¿Qué acaba de pasar?

Todavía no hay ningún LLM. Solo hemos ejecutado una función normal de Python.

El resultado esperado es:

```python
['09:30 Mentoría', '18:00 Bootcamp']
```

Ahora vamos a convertir esa función en una herramienta que nuestro orquestador pueda seleccionar.

## 2. Registramos las herramientas permitidas

El orquestador necesita una lista controlada de funciones que sí puede ejecutar. Vamos a usar un diccionario como **registro de herramientas** o *allowlist*.

La clave será el nombre que utilizará la tool call y el valor será la función Python real.

In [ ]:
herramientas = {
    "consultar_agenda": None  # COMPLETA: sustituye None por el nombre de la función
}

assert callable(herramientas["consultar_agenda"]), (
    "Debes guardar la función consultar_agenda sin paréntesis."
)

print("Herramientas registradas:", list(herramientas.keys()))

### ¿Por qué sin paréntesis?

- `consultar_agenda` → representa la función y podemos guardarla.
- `consultar_agenda(...)` → ejecutaría la función inmediatamente.

Aquí queremos guardar la herramienta para ejecutarla **más tarde**, cuando el sistema la necesite.

## 3. Simulamos una tool call

Un modelo no tiene que ejecutar la función directamente. Puede devolver una solicitud estructurada con:

- `name`: qué herramienta quiere usar;
- `arguments`: qué valores necesita esa herramienta.

Completa únicamente el nombre de la tool.

In [ ]:
tool_call = {
    "name": "",  # COMPLETA AQUÍ
    "arguments": {
        "fecha": "2026-10-26"
    }
}

assert tool_call["name"] == "consultar_agenda", (
    "El nombre debe coincidir exactamente con la clave registrada: consultar_agenda"
)

print(tool_call)

### Observa los argumentos

Tenemos:

```python
{"fecha": "2026-10-26"}
```

La clave se llama `fecha` porque nuestra función fue definida así:

```python
def consultar_agenda(fecha):
```

Este detalle es fundamental cuando utilicemos `**argumentos`.

## 4. Extraemos y validamos la petición

Ahora sacamos de la tool call el nombre de la herramienta y sus argumentos.

Después comprobaremos si la herramienta solicitada está realmente autorizada.

In [ ]:
nombre = tool_call["name"]
argumentos = tool_call["arguments"]

print("Nombre:", nombre)
print("Argumentos:", argumentos)

### Ejercicio 2 · Validar la herramienta

Completa `registro_permitido` usando el diccionario que contiene las herramientas autorizadas.

**Pista:** lo creamos en el apartado 2.

In [ ]:
registro_permitido = None  # COMPLETA AQUÍ

assert registro_permitido is herramientas, (
    "Usa el diccionario herramientas que creamos antes."
)

if nombre in registro_permitido:
    print("Tool autorizada")
else:
    print("Tool no autorizada")

### Seguridad

El modelo puede **solicitar** una herramienta, pero no decide por sí solo si puede ejecutarla.

Nuestro software comprueba si el nombre recibido está dentro del registro permitido. Esta separación es una de las ideas centrales de Tool Use.

## 5. Seleccionamos la función correcta

Ya sabemos que `nombre` vale `"consultar_agenda"` y que esa herramienta está autorizada.

Ahora recuperaremos del diccionario la función Python asociada a ese nombre.

In [ ]:
funcion = herramientas[nombre]

assert callable(funcion), "La herramienta seleccionada debe ser una función ejecutable."
print("Función seleccionada:", funcion.__name__)

## 6. Ejecutamos usando `**argumentos`

Aquí aparece una de las partes más importantes.

Si tenemos:

```python
argumentos = {"fecha": "2026-10-26"}
```

entonces:

```python
funcion(**argumentos)
```

es equivalente a:

```python
funcion(fecha="2026-10-26")
```

### Ejercicio 3
Completa `argumentos_para_funcion` con el diccionario que ya contiene los argumentos de la tool call.

In [ ]:
argumentos_para_funcion = None  # COMPLETA AQUÍ

assert argumentos_para_funcion == argumentos, (
    "Usa la variable argumentos creada a partir de tool_call['arguments']."
)
assert set(argumentos_para_funcion.keys()) == {"fecha"}, (
    "La función consultar_agenda espera un argumento llamado fecha."
)

resultado = funcion(**argumentos_para_funcion)
print(resultado)

### Resultado del flujo

Acabamos de hacer esto:

1. La tool call pidió `consultar_agenda`.
2. El software comprobó que estaba autorizada.
3. Recuperó la función correspondiente.
4. Pasó `fecha="2026-10-26"` mediante `**argumentos`.
5. La función devolvió los eventos.

Eso ya es el núcleo de un sistema de **Tool Use**.

## 7. Añadimos una segunda herramienta

Un sistema real suele tener más de una tool. Añadiremos una función sencilla para multiplicar dos números.

In [ ]:
def multiplicar(a, b):
    return a * b

herramientas["multiplicar"] = None  # COMPLETA AQUÍ

assert callable(herramientas["multiplicar"]), (
    "Guarda la función multiplicar sin paréntesis."
)

print("Herramientas disponibles:", list(herramientas.keys()))

## 8. Segunda tool call

Queremos calcular **24 × 17**.

Completa únicamente:

- el nombre de la herramienta;
- el valor de `a`;
- el valor de `b`.

In [ ]:
tool_call_2 = {
    "name": "",   # COMPLETA
    "arguments": {
        "a": 0,   # COMPLETA
        "b": 0    # COMPLETA
    }
}

assert tool_call_2["name"] == "multiplicar", (
    "La segunda tool call debe solicitar la herramienta multiplicar."
)
assert tool_call_2["arguments"] == {"a": 24, "b": 17}, (
    "Los argumentos deben representar 24 × 17."
)

print(tool_call_2)

## 9. Reutilizamos el mismo patrón

Fíjate en que no necesitamos inventar otro sistema. El mismo flujo sirve para cualquier herramienta registrada.

In [ ]:
nombre2 = tool_call_2["name"]
argumentos2 = tool_call_2["arguments"]

assert nombre2 in herramientas, "Tool no autorizada."

funcion2 = herramientas[nombre2]
resultado2 = funcion2(**argumentos2)

print("Resultado:", resultado2)
assert resultado2 == 408, "El resultado esperado de 24 × 17 es 408."
print("✓ Segunda tool ejecutada correctamente")

## 10. ¿Qué ocurre si una tool no está autorizada?

Vamos a simular una petición peligrosa o simplemente inexistente. Esta vez **no debes completar nada**: observa cómo el software la bloquea antes de intentar ejecutarla.

In [ ]:
tool_call_no_autorizada = {
    "name": "borrar_base_de_datos",
    "arguments": {}
}

nombre_bloqueado = tool_call_no_autorizada["name"]

if nombre_bloqueado in herramientas:
    print("Se podría ejecutar")
else:
    print("Bloqueada: tool no autorizada")

### ¿Por qué es importante?

Tool Use no significa dar al LLM acceso libre a cualquier función del programa.

En un sistema bien diseñado:

- se define qué tools existen;
- se validan los nombres;
- se validan los argumentos;
- se controlan los permisos;
- y algunas acciones sensibles pueden exigir confirmación humana.

# Comprobación final

Ejecuta esta celda cuando hayas completado todos los ejercicios. Si todo está bien, verás **“Práctica completada correctamente”**.

In [ ]:
assert fecha_consulta == "2026-10-26"
assert herramientas["consultar_agenda"] is consultar_agenda
assert tool_call["name"] == "consultar_agenda"
assert nombre in herramientas
assert argumentos == {"fecha": "2026-10-26"}
assert resultado == ["09:30 Mentoría", "18:00 Bootcamp"]
assert herramientas["multiplicar"] is multiplicar
assert tool_call_2 == {
    "name": "multiplicar",
    "arguments": {"a": 24, "b": 17}
}
assert resultado2 == 408

print("✓ Práctica completada correctamente")

# Lo que debes llevarte

Has construido un flujo de Tool Use sin necesidad de conectar todavía un LLM real:

```text
Solicitud estructurada
        ↓
Validar nombre
        ↓
Buscar función permitida
        ↓
Validar / preparar argumentos
        ↓
Ejecutar función
        ↓
Obtener resultado
```

En una integración real con un LLM cambia **quién genera la `tool_call`**, pero el trabajo del software sigue siendo esencialmente el mismo.

> **El modelo decide qué herramienta necesita. El software valida y ejecuta la herramienta.**